In [13]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
TCGA 外部验证数据 —— 样本内配准 + 薄层(155层)转厚层(24层)
策略：仅做 T2 与 T1GD 之间的仿射配准，不裁剪背景，不配准到外部模板
"""

import os
import sys
import numpy as np
import nibabel as nib
import ants
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
import shutil

# ==================== 配置参数 ====================
# --- 输入输出路径 ---
TCGA_DATA_ROOT = r"E:\NETS\bulk&影像 TCGA外部验证\原数据\1_TCGA_Filtered_Data\1_TCGA_Imaging_T1Gd_T2_Axial_Sagittal"
OUTPUT_ROOT = r"E:\NETS\TCGA_processed_internal"          # 输出根目录
QC_COLLECT_DIR = os.path.join(OUTPUT_ROOT, "QC_collection")

# --- 关键词识别（根据你的实际文件命名调整）---
T2_KEYWORDS = ['T2', 't2']                # 匹配 T2 文件
CE_KEYWORDS = ['T1GD', 'T1Gd', 'CE', 'ce'] # 匹配增强序列（T1Gd）

# --- 重采样与配准参数 ---
TARGET_XY = (1.0, 1.0)                # XY 重采样到 1mm
TARGET_Z_SLICES = 24                  # 目标层数（厚层）
REG_TYPE = 'Affine'                   # 仿射配准（适用于同一患者的脑部）
DO_NORMALIZE = True                   # 是否做 Z-score 标准化
SAVE_SLICES = True                    # 是否保存每层切片 JPG

# 掩膜：明确不做（保留完整头部）
CROP_AND_MASK = False

# ==================== 辅助函数 ====================
def safe_read_ants(path):
    """尝试用 ANTs 读取，失败则通过 nibabel 中转"""
    try:
        return ants.image_read(path)
    except Exception:
        print(f"   [警告] ants读取失败，使用nibabel中转: {Path(path).name}")
        nib_img = nib.load(path)
        temp_dir = Path(os.environ.get("TEMP", "."))
        temp_path = temp_dir / f"temp_ants_{Path(path).stem}.nii"
        nib.save(nib_img, temp_path)
        ants_img = ants.image_read(str(temp_path))
        temp_path.unlink()
        return ants_img

def find_file_by_keywords(folder_path, keywords):
    """在文件夹中查找包含任一关键词的 NIfTI 文件"""
    folder = Path(folder_path)
    for ext in ['*.nii.gz', '*.nii']:
        for file in folder.glob(ext):
            if any(kw.lower() in file.name.lower() for kw in keywords):
                return str(file)
    return None

def resample_to_24_slices(img, target_xy, target_slices):
    """
    将 3D 图像重采样到指定的 XY 分辨率和 Z 轴层数（厚层）
    保持物理视野大小不变，只改变体素数
    """
    orig_spacing = img.spacing
    orig_shape = img.shape
    
    # 计算新的 Z 轴间距，使得 Z 轴体素数恰好变为 target_slices
    new_spacing_z = (orig_spacing[2] * orig_shape[2]) / target_slices
    new_spacing = (target_xy[0], target_xy[1], new_spacing_z)
    
    # 重采样（use_voxels=False 表示基于物理空间，而非基于体素网格）
    resampled = ants.resample_image(img, new_spacing, use_voxels=False, interp_type=1)
    
    # 由于数值误差，层数可能略偏，强制截断或插值到精确层数（这里通常很准）
    # 若不放心，可以在这里做一次强制 resize，但 ANTs 的精度已经足够
    return resampled

def normalize_image(img):
    """Z-score 标准化（全局）"""
    data = img.numpy()
    data = np.nan_to_num(data)
    mean, std = data.mean(), data.std()
    if std > 0:
        data = (data - mean) / std
    return ants.from_numpy(data, origin=img.origin, spacing=img.spacing, direction=img.direction)

def generate_overlay_qc(t2_img, ce_img, output_jpg):
    """生成 T2 和 CE 中层叠加的 QC 图"""
    t2_data = t2_img.numpy()
    ce_data = ce_img.numpy()
    mid = t2_data.shape[2] // 2
    t2_slice = t2_data[:, :, mid].T
    ce_slice = ce_data[:, :, mid].T
    t2_norm = (t2_slice - t2_slice.min()) / (t2_slice.max() - t2_slice.min() + 1e-8)
    ce_norm = (ce_slice - ce_slice.min()) / (ce_slice.max() - ce_slice.min() + 1e-8)
    overlay = np.stack([t2_norm, ce_norm, np.zeros_like(t2_norm)], axis=-1)
    plt.figure(figsize=(6,6))
    plt.imshow(overlay)
    plt.axis('off')
    plt.savefig(output_jpg, bbox_inches='tight', pad_inches=0, dpi=150)
    plt.close()

def save_all_slices_as_jpg(img, base_dir, prefix, modality):
    """保存所有轴向切片为 JPG"""
    data = img.numpy()
    n_slices = data.shape[2]
    slice_dir = os.path.join(base_dir, f"{modality}_slices")
    os.makedirs(slice_dir, exist_ok=True)
    data_min, data_max = data.min(), data.max()
    if data_max - data_min == 0:
        data_norm = np.zeros_like(data)
    else:
        data_norm = (data - data_min) / (data_max - data_min)
    for z in range(n_slices):
        slice_2d = data_norm[:, :, z].T
        plt.figure(figsize=(6,6))
        plt.imshow(slice_2d, cmap='gray', vmin=0, vmax=1)
        plt.axis('off')
        jpg_name = f"{prefix}_slice_{z:03d}.jpg"
        jpg_path = os.path.join(slice_dir, jpg_name)
        plt.savefig(jpg_path, bbox_inches='tight', pad_inches=0, dpi=150)
        plt.close()
    print(f"   已保存 {n_slices} 张 {modality} 图层")

def process_sample(ce_path, t2_path, sample_name, output_root):
    """
    核心处理函数：
    1. T2 和 CE 分别重采样到 24 层（厚层）
    2. CE 配准到 T2（样本内仿射）
    3. 标准化，输出
    """
    sample_dir = os.path.join(output_root, sample_name)
    os.makedirs(sample_dir, exist_ok=True)
    print(f"\n处理样本: {sample_name}")

    # 1. 读取原始图像
    t2_orig = safe_read_ants(t2_path)
    ce_orig = safe_read_ants(ce_path)
    print(f"   原始 T2 层数: {t2_orig.shape[2]}, 原始 CE 层数: {ce_orig.shape[2]}")

    # 2. 分别重采样为 24 层（保持 XY 为 1mm）
    t2_resampled = resample_to_24_slices(t2_orig, TARGET_XY, TARGET_Z_SLICES)
    ce_resampled = resample_to_24_slices(ce_orig, TARGET_XY, TARGET_Z_SLICES)
    print(f"   重采样后 T2 层数: {t2_resampled.shape[2]}, CE 层数: {ce_resampled.shape[2]}")

    # 如果层数因舍入误差不是 24，强制裁剪/插值到 24（极少发生，以防万一）
    if t2_resampled.shape[2] != TARGET_Z_SLICES:
        print(f"   ⚠️ T2 层数为 {t2_resampled.shape[2]}，强制重采样到 {TARGET_Z_SLICES}")
        t2_resampled = ants.resample_image(t2_resampled, (TARGET_XY[0], TARGET_XY[1], t2_resampled.spacing[2] * t2_resampled.shape[2] / TARGET_Z_SLICES), use_voxels=False)
    if ce_resampled.shape[2] != TARGET_Z_SLICES:
        print(f"   ⚠️ CE 层数为 {ce_resampled.shape[2]}，强制重采样到 {TARGET_Z_SLICES}")
        ce_resampled = ants.resample_image(ce_resampled, (TARGET_XY[0], TARGET_XY[1], ce_resampled.spacing[2] * ce_resampled.shape[2] / TARGET_Z_SLICES), use_voxels=False)

    # 3. 样本内配准：以 T2 为固定图像，CE 为浮动图像（仿射）
    print("   执行 CE 到 T2 的仿射配准...")
    reg = ants.registration(fixed=t2_resampled, moving=ce_resampled, type_of_transform=REG_TYPE)
    ce_warped = reg['warpedmovout']
    
    # 4. 标准化（Z-score）
    if DO_NORMALIZE:
        t2_final = normalize_image(t2_resampled)
        ce_final = normalize_image(ce_warped)
    else:
        t2_final = t2_resampled
        ce_final = ce_warped

    # 5. 保存 NIfTI 文件
    t2_out = os.path.join(sample_dir, f"{sample_name}_T2_final.nii.gz")
    ce_out = os.path.join(sample_dir, f"{sample_name}_CE_final.nii.gz")
    ants.image_write(t2_final, t2_out)
    ants.image_write(ce_final, ce_out)

    # 6. 生成 QC 叠加图（中层）
    qc_path = os.path.join(sample_dir, f"{sample_name}_overlay_QC.jpg")
    generate_overlay_qc(t2_final, ce_final, qc_path)

    # 7. 保存所有轴向切片 JPG（共 24 层）
    if SAVE_SLICES:
        save_all_slices_as_jpg(t2_final, sample_dir, f"{sample_name}_T2_final", "T2")
        save_all_slices_as_jpg(ce_final, sample_dir, f"{sample_name}_CE_final", "CE")

    print(f"✓ 完成 {sample_name}")
    return qc_path

# ==================== 主程序 ====================
if __name__ == "__main__":
    print("="*70)
    print("TCGA 外部验证数据 —— 样本内配准 + 薄层转厚层(24层)")
    print("配准方式: CE -> T2 (仿射)，不裁剪掩膜，不配准外部模板")
    print("="*70)

    # 创建输出目录
    os.makedirs(OUTPUT_ROOT, exist_ok=True)
    os.makedirs(QC_COLLECT_DIR, exist_ok=True)

    # 遍历 TCGA 数据根目录下的所有子文件夹
    root = Path(TCGA_DATA_ROOT)
    if not root.exists():
        print(f"❌ 数据目录不存在: {TCGA_DATA_ROOT}")
        sys.exit(1)

    sample_folders = [f for f in root.iterdir() if f.is_dir()]
    print(f"发现 {len(sample_folders)} 个病例文件夹\n")

    success_count = 0
    for folder in tqdm(sample_folders, desc="处理进度"):
        sample_name = folder.name
        # 查找 T2 和 CE 文件
        t2_file = find_file_by_keywords(folder, T2_KEYWORDS)
        ce_file = find_file_by_keywords(folder, CE_KEYWORDS)
        if not t2_file or not ce_file:
            print(f"⚠️ 跳过 {sample_name}: 缺少 T2 或 CE 文件 (T2:{t2_file}, CE:{ce_file})")
            continue
        try:
            qc_path = process_sample(ce_file, t2_file, sample_name, OUTPUT_ROOT)
            if qc_path:
                # 复制 QC 图到收集文件夹
                dest = os.path.join(QC_COLLECT_DIR, f"{sample_name}_overlay_QC.jpg")
                shutil.copy2(qc_path, dest)
                success_count += 1
        except Exception as e:
            print(f"❌ 处理 {sample_name} 出错: {e}")

    print("\n" + "="*70)
    print(f"处理完成！成功: {success_count}, 总计: {len(sample_folders)}")
    print(f"输出目录: {OUTPUT_ROOT}")
    print(f"QC 图汇总: {QC_COLLECT_DIR}")
    print("="*70)

TCGA 外部验证数据 —— 样本内配准 + 薄层转厚层(24层)
配准方式: CE -> T2 (仿射)，不裁剪掩膜，不配准外部模板
发现 85 个病例文件夹



处理进度:   0%|          | 0/85 [00:00<?, ?it/s]


处理样本: TCGA-02-0047
   [警告] ants读取失败，使用nibabel中转: TCGA-02-0047_1998.12.15_t2.nii.gz
   [警告] ants读取失败，使用nibabel中转: TCGA-02-0047_1998.12.15_t1Gd.nii.gz
   原始 T2 层数: 155, 原始 CE 层数: 155
   重采样后 T2 层数: 24, CE 层数: 24
   执行 CE 到 T2 的仿射配准...
   已保存 24 张 T2 图层


处理进度:   1%|          | 1/85 [00:12<17:25, 12.45s/it]

   已保存 24 张 CE 图层
✓ 完成 TCGA-02-0047

处理样本: TCGA-06-0130
   [警告] ants读取失败，使用nibabel中转: TCGA-06-0130_2001.09.11_t2.nii.gz
   [警告] ants读取失败，使用nibabel中转: TCGA-06-0130_2001.09.11_t1Gd.nii.gz
   原始 T2 层数: 155, 原始 CE 层数: 155
   重采样后 T2 层数: 24, CE 层数: 24
   执行 CE 到 T2 的仿射配准...
   已保存 24 张 T2 图层


...

   已保存 24 张 CE 图层
✓ 完成 TCGA-HT-8114

处理样本: TCGA-HT-8563
   [警告] ants读取失败，使用nibabel中转: TCGA-HT-8563_1998.12.09_t2.nii.gz
   [警告] ants读取失败，使用nibabel中转: TCGA-HT-8563_1998.12.09_t1Gd.nii.gz
   原始 T2 层数: 155, 原始 CE 层数: 155
   重采样后 T2 层数: 24, CE 层数: 24
   执行 CE 到 T2 的仿射配准...
   已保存 24 张 T2 图层


处理进度:  99%|█████████▉| 84/85 [18:13<00:12, 12.24s/it]

   已保存 24 张 CE 图层
✓ 完成 TCGA-HT-8563

处理样本: TCGA-HT-A61A
   [警告] ants读取失败，使用nibabel中转: TCGA-HT-A61A_2000.01.27_t2.nii.gz
   [警告] ants读取失败，使用nibabel中转: TCGA-HT-A61A_2000.01.27_t1Gd.nii.gz
   原始 T2 层数: 155, 原始 CE 层数: 155
   重采样后 T2 层数: 24, CE 层数: 24
   执行 CE 到 T2 的仿射配准...
   已保存 24 张 T2 图层


处理进度: 100%|██████████| 85/85 [18:26<00:00, 13.02s/it]

   已保存 24 张 CE 图层
✓ 完成 TCGA-HT-A61A

处理完成！成功: 85, 总计: 85
输出目录: E:\NETS\TCGA_processed_internal
QC 图汇总: E:\NETS\TCGA_processed_internal\QC_collection
